# Session 06, Part 2 — Async subagents

A **sync** subagent (Part 1, `task`) blocks the main agent until it returns. An **async** subagent runs as a
background job: the supervisor gets a **task ID** right away, keeps talking with the user, and later checks,
updates or cancels the task. Deep Agents runs async subagents on an **Agent Server** (the Agent Protocol: threads
and runs).

<img src="images/b6_sync_vs_async.png" width="640">

Here both agents live in **one deployment**: [`langgraph.json`](../langgraph.json) registers the graphs
`supervisor` ([`app/supervisor.py`](../app/supervisor.py)) and `researcher` ([`app/researcher.py`](../app/researcher.py)).
Because they share the server, the `AsyncSubAgent` needs no `url`: the supervisor reaches the researcher in-process
(ASGI transport), which the docs recommend as the default.

| Part | What you see |
|---|---|
| 1 | Launch two tasks in one message, get task IDs in seconds, ask something else while they run, watch them run, collect results |
| 2 | Update a running task with new instructions, cancel another, list all tasks |
| 3 | Budget: the limits that stop a supervisor from launching or polling too much |

Docs: [Deep Agents async subagents](https://docs.langchain.com/oss/python/deepagents/async-subagents)

In [1]:
import logging
import os

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

# ── Keys (.env of this session folder; the server reads the same file) ───────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
logger.info("Model: %s", os.getenv("MODEL", "openai:gpt-6-luna"))

INFO | Model: openai:gpt-6-luna


## Start the Agent Server

This cell starts `langgraph dev` (a local, in-memory Agent Server on port 2024) from the session folder and waits
until it answers. `--n-jobs-per-worker 4` gives 4 run slots: 1 for the supervisor turn, 3 for researcher tasks; a
fifth run waits in the queue. Server logs go to `output/langgraph_dev.log`. The last cell of the notebook stops it.

In [2]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

import httpx

SESSION_DIR = Path.cwd().parent
SERVER = "http://127.0.0.1:2024"
Path("output").mkdir(exist_ok=True)
langgraph_cli = shutil.which("langgraph", path=str(Path(sys.executable).parent))  # from langgraph-cli[inmem] in venvAE06
server_process = subprocess.Popen(
    [langgraph_cli, "dev", "--no-browser", "--port", "2024", "--n-jobs-per-worker", "4"],
    cwd=SESSION_DIR, stdout=open("output/langgraph_dev.log", "w"), stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        if httpx.get(f"{SERVER}/ok").status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Agent Server did not start: read notebooks/output/langgraph_dev.log")
logger.info("Agent Server: ok (graphs supervisor, researcher)")

INFO | Agent Server: ok (graphs supervisor, researcher)


## Talking to the supervisor

The notebook is only a client of the server (`langgraph_sdk`): one **thread** is the conversation, one **run** is a
turn. `turn()` streams a run and prints one line per tool call, then the answer and the task table that the
supervisor keeps in its `async_tasks` state channel. `watch()` asks the server directly for the latest run of each
task (no model call, no cost): the **task ID is the thread ID** of the researcher's run.

In [3]:
import asyncio
import json

from langgraph_sdk import get_client

server = get_client(url=SERVER)
thread_id = (await server.threads.create())["thread_id"]


def short(text, n: int = 100) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n] + "…"


async def turn(text: str) -> dict:
    """One user turn: stream the supervisor's run, then print time, answer and the task table."""
    print(f"USER: {text}")
    started, answer = time.monotonic(), ""
    async for chunk in server.runs.stream(thread_id, "supervisor", stream_mode="updates",
                                          input={"messages": [{"role": "user", "content": text}]}):
        for update in (chunk.data or {}).values() if chunk.event == "updates" else []:
            for m in (update or {}).get("messages", []) if isinstance(update, dict) else []:
                for c in m.get("tool_calls") or []:
                    print(f"[supervisor] → {c['name']}({short(json.dumps(c['args'], ensure_ascii=False), 80)})")
                if m["type"] == "tool":
                    print(f"[supervisor] ← {m['name']}: {short(m['content'])}")
                elif m["type"] == "ai" and not m.get("tool_calls"):
                    content = m["content"]
                    answer = content if isinstance(content, str) else "".join(b.get("text", "") for b in content)
    tasks = (await server.threads.get_state(thread_id))["values"].get("async_tasks", {})
    print(answer)
    print(f"→ turn took {time.monotonic() - started:.1f} s | tasks: "
          + (", ".join(f"{tid[-6:]} {t['status']}" for tid, t in tasks.items()) or "none"))
    return tasks


async def watch(tasks: dict, every: int = 10, timeout: int = 180) -> None:
    """Latest run status of each task, read from the server until none is pending or running."""
    started = time.monotonic()
    while True:
        runs = {tid: (await server.runs.list(tid))[0]["status"] for tid in tasks}
        print(f"{time.monotonic() - started:5.0f} s  " + "  ".join(f"{tid[-6:]}: {s}" for tid, s in runs.items()))
        if all(s not in ("pending", "running") for s in runs.values()) or time.monotonic() - started > timeout:
            return
        await asyncio.sleep(every)

## Part 1 — Launch, keep talking, collect

Two independent questions: the supervisor calls `start_async_task` twice in one message and ends its turn. Each
researcher pauses `RESEARCH_DELAY_S` = 15 s before each search (`app/researcher.py`), so the tasks stay `running`
long enough to watch; set it to 0 for a real-speed run.

In [4]:
tasks = await turn("Research in the background: (1) how well air-source heat pumps work in freezing weather, "
                   "(2) how long electric-car batteries usually last.")

USER: Research in the background: (1) how well air-source heat pumps work in freezing weather, (2) how long electric-car batteries usually last.
[supervisor] → start_async_task({"description": "Research how well modern air-source heat pumps work in freezing…)
[supervisor] → start_async_task({"description": "Research how long electric-car batteries usually last, includin…)
[supervisor] ← start_async_task: Launched async subagent. task_id: 01a1551a-41c7-71c1-b5f0-7cc9fd74e668
[supervisor] ← start_async_task: Launched async subagent. task_id: 01a1551a-41c8-7db0-bc82-d2bbb932b222
Launched two background research tasks:

1. Air-source heat pumps in freezing weather — task ID `01a1551a-41c7-71c1-b5f0-7cc9fd74e668`
2. Typical electric-car battery lifespan — task ID `01a1551a-41c8-7db0-bc82-d2bbb932b222`
→ turn took 8.1 s | tasks: 74e668 running, 32b222 running


The turn ended while both researchers are still working. The supervisor is free: the next question is answered
directly, without waiting for the tasks.

In [5]:
_ = await turn("While they run: what does COP mean for a heat pump, in one sentence?")

USER: While they run: what does COP mean for a heat pump, in one sentence?
COP (coefficient of performance) is the ratio of heat a heat pump delivers to the electricity it uses; a COP of 3 means it provides three units of heat per unit of electricity.
→ turn took 3.7 s | tasks: 74e668 running, 32b222 running


In [6]:
await watch(tasks)

    0 s  74e668: running  32b222: running
   10 s  74e668: running  32b222: running
   20 s  74e668: running  32b222: running
   30 s  74e668: running  32b222: success
   40 s  74e668: running  32b222: success
   50 s  74e668: running  32b222: success
   60 s  74e668: success  32b222: success


In [7]:
_ = await turn("Both tasks should be done now. Check them and give me the results.")

USER: Both tasks should be done now. Check them and give me the results.
[supervisor] → check_async_task({"task_id": "01a1551a-41c7-71c1-b5f0-7cc9fd74e668"})
[supervisor] → check_async_task({"task_id": "01a1551a-41c8-7db0-bc82-d2bbb932b222"})
[supervisor] ← check_async_task: {"status": "success", "thread_id": "01a1551a-41c7-71c1-b5f0-7cc9fd74e668", "result": [{"id": "rs_0e5…
[supervisor] ← check_async_task: {"status": "success", "thread_id": "01a1551a-41c8-7db0-bc82-d2bbb932b222", "result": [{"id": "rs_026…
### Air-source heat pumps in freezing weather
Modern **cold-climate** heat pumps can keep heating in subfreezing conditions, though their efficiency and heat output generally fall as outdoor temperatures drop. One report on the U.S. Department of Energy’s Cold Climate Heat Pump Challenge says tested systems maintained over 70% of their heating capacity at **−15°F (−26°C)**; that’s a result for those systems, not a guarantee for all models. The right choice depends on the specific un

## Part 2 — Update and cancel running tasks

`update_async_task` sends new instructions to the same task (a new run on the same thread, which interrupts the old
one); `cancel_async_task` stops it. A cancelled run shows as `interrupted` on the server. A sync subagent offers
neither.

In [8]:
tasks = await turn("Start two more background tasks: rooftop solar payback time, and hydrogen fuel-cell cars.")

USER: Start two more background tasks: rooftop solar payback time, and hydrogen fuel-cell cars.
[supervisor] → start_async_task({"description": "Research typical rooftop solar payback time for residential sys…)
[supervisor] → start_async_task({"description": "Research hydrogen fuel-cell cars: how they work, current availa…)
[supervisor] ← start_async_task: Launched async subagent. task_id: 01a1551b-873e-7691-ab8b-a72b14dddc0a
[supervisor] ← start_async_task: Launched async subagent. task_id: 01a1551b-873f-7393-babf-d912c21b1166
Launched two background research tasks:

1. Rooftop solar payback time — task ID `01a1551b-873e-7691-ab8b-a72b14dddc0a`
2. Hydrogen fuel-cell cars — task ID `01a1551b-873f-7393-babf-d912c21b1166`
→ turn took 7.6 s | tasks: 74e668 success, 32b222 success, dddc0a running, 1b1166 running


In [9]:
_ = await turn("Update the solar task: only consider Germany. Cancel the hydrogen task, we no longer need it.")

USER: Update the solar task: only consider Germany. Cancel the hydrogen task, we no longer need it.
[supervisor] → update_async_task({"task_id": "01a1551b-873e-7691-ab8b-a72b14dddc0a", "message": "Please restrict …)
[supervisor] → cancel_async_task({"task_id": "01a1551b-873f-7393-babf-d912c21b1166"})
[supervisor] ← cancel_async_task: Cancelled async subagent task: 01a1551b-873f-7393-babf-d912c21b1166
[supervisor] ← update_async_task: Updated async subagent. task_id: 01a1551b-873e-7691-ab8b-a72b14dddc0a
Updated the rooftop-solar task to focus exclusively on Germany. Cancelled the hydrogen fuel-cell car task.
→ turn took 7.0 s | tasks: 74e668 success, 32b222 success, dddc0a running, 1b1166 cancelled


In [10]:
await watch(tasks)
_ = await turn("List all tasks with their current status.")

    0 s  74e668: success  32b222: success  dddc0a: running  1b1166: interrupted
   10 s  74e668: success  32b222: success  dddc0a: running  1b1166: interrupted
   20 s  74e668: success  32b222: success  dddc0a: running  1b1166: interrupted
   30 s  74e668: success  32b222: success  dddc0a: running  1b1166: interrupted
   40 s  74e668: success  32b222: success  dddc0a: success  1b1166: interrupted
USER: List all tasks with their current status.
[supervisor] → list_async_tasks({"status_filter": "all"})
[supervisor] ← list_async_tasks: 4 tracked task(s): - task_id: 01a1551a-41c7-71c1-b5f0-7cc9fd74e668 agent: researcher status: success…
- **Air-source heat pumps in freezing weather** — Complete (`01a1551a-41c7-71c1-b5f0-7cc9fd74e668`)
- **Electric-car battery lifespan** — Complete (`01a1551a-41c8-7db0-bc82-d2bbb932b222`)
- **Rooftop solar payback in Germany** — Complete (`01a1551b-873e-7691-ab8b-a72b14dddc0a`)
- **Hydrogen fuel-cell cars** — Cancelled (`01a1551b-873f-7393-babf-d912c21b1166

## Part 3 — Budget: launches and checks are capped

The supervisor already launched 4 tasks in this thread, the `thread_limit` of `start_async_task` in
`app/supervisor.py`. A fifth launch is refused before any background run starts. Together with the caps inside
`app/researcher.py` (4 model calls, 2 searches per run) and the 4 server slots, the worst case is known in advance.

In [11]:
_ = await turn("Also research e-bike battery prices in the background.")

USER: Also research e-bike battery prices in the background.
[supervisor] → start_async_task({"description": "Research current prices for replacement batteries for electric …)
[supervisor] ← start_async_task: Tool call limit exceeded. Do not call 'start_async_task' again.
I couldn’t start the e-bike battery price research task because the background-task tool refused the request due to a call limit. No task was started.
→ turn took 7.7 s | tasks: 74e668 success, 32b222 success, dddc0a success, 1b1166 cancelled


To drive the supervisor yourself (launch, ask something else, check, update, cancel), uncomment the cell below.
Every launch still counts toward the limits above.

In [12]:
# while (text := input("You (empty to stop): ").strip()):
#     await turn(text)

Stop the Agent Server (its threads and tasks are in memory and disappear).

In [13]:
server_process.terminate()
server_process.wait()
logger.info("Agent Server stopped")

INFO | Agent Server stopped
